## Imports and Inititalizations

In [ ]:
import pandas as pd
import base64
import os
import ollama
from io import BytesIO
from PIL import Image
import nltk
from nltk.metrics import edit_distance
import json

# Evaluator used in CC_OCR
from doc_parsing_evaluator import ParsingEvaluator

# Ensure NLTK data is available for edit distance
nltk.download('punkt')

# Paths based on your local directory structure
BASE_DIR = r"D:\Projects\BTP\CC-OCR_Dataset\doc_parsing"
OUT_DIR = r"D:\Projects\BTP\Evaluation_Results"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = "qwen2.5vl:3b"

# Initialize the evaluator
evaluator = ParsingEvaluator(group_name="OCR_Eval")

## Inference Engine (Ollama)

In [ ]:
def query_qwen(image_bytes, task_type):
    """Sends image to Ollama with format-specific instructions """
    if task_type == "table":
        prompt = "Parse this table into a clean, structured HTML format with <table> tags. Include all row and column spans."
    else:
        # Document track uses LaTeX for text and formatting
        prompt = "Parse this document into LaTeX format. Provide only the LaTeX code."

    img_b64 = base64.b64encode(image_bytes).decode('utf-8')
    
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[{
            'role': 'user',
            'content': prompt,
            'images': [img_b64]
        }]
    )
    return response['message']['content']

## Document Parsing Evaluation (LaTeX)

In [ ]:
doc_tasks = {
    "Scanned": os.path.join(BASE_DIR, "doc", "doc_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "doc", "doc_photo_eng_75.tsv")
}

doc_results = {}

for label, path in doc_tasks.items():
    if not os.path.exists(path): continue
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} documents...")
    
    for _, row in df.iterrows():
        img_bytes = base64.b64decode(row['image'])
        ground_truth = str(row['answer']).strip() 
        
        prediction = query_qwen(img_bytes, "doc")
        
        # FIX: Use the professional NED logic with LaTeX cleaning
        score = evaluator.evaluate_single_doc_sample(ground_truth, prediction)
        scores.append(score)
        
    doc_results[label] = sum(scores) / len(scores) if scores else 0

print(f"Document Parsing (NED with LaTeX) Results: {doc_results}")

## Table Parsing Evaluation (HTML)

In [ ]:
table_tasks = {
    "Scanned": os.path.join(BASE_DIR, "table", "table_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "table", "table_photo_eng_75.tsv")
}

table_results = {}

for label, path in table_tasks.items():
    if not os.path.exists(path): 
        print(f"Skipping {label}: Path not found.")
        continue
        
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} tables...")
    
    for _, row in df.iterrows():
        img_bytes = base64.b64decode(row['image'])
        ground_truth_html = str(row['answer']).strip() 
        
        prediction = query_qwen(img_bytes, "table")
        
        # FIX: Use the official Tree Edit Distance (TEDS) metric
        score = evaluator.evaluate_single_table_sample(ground_truth_html, prediction)
        scores.append(score)
        
    table_results[label] = sum(scores) / len(scores) if scores else 0

print(f"\nTable Parsing (TEDS Results): {table_results}")

## Final Report 

In [ ]:
summary = {
    "Document Parsing (NED-LaTeX)": doc_results,
    "Table Parsing (HTML-Sim)": table_results,
    "Model": MODEL_NAME,
    "Track": "English Document Parsing"
}

with open(os.path.join(OUT_DIR, "stage1_final_report.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("Evaluation Complete. Check 'stage1_final_report.json' for full details.")